# 03 - Comparação de Votação em Leituras Longas e Diagnóstico da Queda de ConvNeXt

Este notebook reproduz a comparação sistemática entre arquiteturas (`long/cnn-long-voter-compare.ipynb`),
gerando os números da **Tabela 12**, estratificando os erros por comprimento, ambiguidade e presença de genoma
(`seen_genome` vs. `unseen_genome`), e diagnosticando formalmente as causas da queda do ConvNeXt (AD-010).

**Invariantes científicas (`AD-010`):**
- Reprodução da **Tabela 12**: ResNet50 atinge 72,69% de acurácia por média de logits, enquanto as variantes de ConvNeXt colapsam (ConvNeXt_Z 12,01%, ConvNeXt_T 3,49%, ConvNeXt_F 19,99%).
- Mecanismos do colapso diagnosticados: (1) viés de amostragem de leituras em vírus gigantes; (2) logit pulling pré-softmax que corrompe o argmax médio; (3) perda de calibração em ConvNeXt_T.


In [ ]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime, reserve_c02_long_run
from viral_cnn.utils.logging_utils import log_artifact_path
from viral_cnn.inference.audit_long import stratify_long_errors
from viral_cnn.baselines.long_eval import evaluate_long_baselines

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
project_root = next(r for r in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (r / "src" / "viral_cnn").exists())
run_id, output_dir = reserve_c02_long_run(project_root, "long/03")
ctx.run_id, ctx.output_dir = run_id, output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


## 2. Reprodução da Tabela 12 do Manuscrito

Consolidação das taxas de acurácia Top-1 por média de logits e por voto majoritário entre as arquiteturas avaliadas.


In [ ]:
# Strictly consume the five artifacts from one explicitly identified long/02 execution.
from viral_cnn.baselines.long_eval import build_c02_table12, C02_LONG_VARIANTS
from viral_cnn.inference.audit_long import stratify_long_errors, build_diagnostic_finding, diagnose_window_logits, compare_query_and_taxon_accuracy, calibrate_mean_logits
run_id=os.getenv("C02_LONG_RUN_ID")
if not run_id: raise RuntimeError("Set C02_LONG_RUN_ID to the shared long/01–03 run namespace")
run_root=project_root/"results_reproduction"/"runs"/run_id
manifest02_path=run_root/"long"/"02"/"manifest_02_voter_inference.json"
manifest01_path=run_root/"long"/"01"/"manifest_01_ncbi_prep.json"
if not manifest02_path.is_file() or not manifest01_path.is_file(): raise FileNotFoundError(f"C02 predecessor manifests missing: {manifest01_path}, {manifest02_path}")
manifest_01=json.loads(manifest01_path.read_text()); manifest_02=json.loads(manifest02_path.read_text())
if bool(manifest_02.get("is_smoke_test")) != bool(ctx.is_smoke_test): raise ValueError("C02 run mode mismatch between current run and predecessor")
artifacts=[]
for variant in C02_LONG_VARIANTS:
    for mode in ("pad","skip"):
        path=Path(run_root/"long"/"02"/f"predictions_{variant}_epsilon0_{mode}.parquet")
        if not path.is_file(): raise FileNotFoundError(f"Missing identified prediction artifact for {variant}/{mode}: {path}")
        log_artifact_path("paired long prediction",path)
        pred_frame=pd.read_parquet(path)
        if mode=="pad":
            logits_path=run_root/"long"/"02"/f"window_logits_sample_{variant}_epsilon0.json"
            if logits_path.is_file():
                logit_data=json.loads(logits_path.read_text()); logit_frame=pd.DataFrame(logit_data["records"]); pred_frame=pred_frame.merge(logit_frame,on="query_id",how="left",validate="one_to_one")
        artifacts.append({"variant":variant,"predictions":pred_frame})
first=artifacts[0]["predictions"]; truth=first[["query_id","truth_label"]].drop_duplicates().copy()
if len(truth)!=len(first): raise ValueError("duplicate query_id in C02 prediction artifact")
tables=[]
for mode in ("pad","skip"):
    tables.append(build_c02_table12([a for a in artifacts if set(a["predictions"]["handle_short"].astype(str))=={mode}],truth))
df_t12=pd.concat(tables,ignore_index=True)
t12_csv=output_dir/"table_12_long_voter_comparison.csv"; df_t12.to_csv(t12_csv,index=False)
table_12_data=df_t12.to_dict(orient="records")


## 3. Estratificação de Erros por Comprimento, Ambiguidade e Presença de Genoma

Estratificação detalhada das previsões para isolar os fatores causais que degradam os modelos.


In [ ]:
# Diagnostics are computed from the same identified query set; missing metadata remains unknown.
df_eval=next(a["predictions"] for a in artifacts if a["variant"]==C02_LONG_VARIANTS[0] and set(a["predictions"]["handle_short"].astype(str))=={"pad"}).copy()
if "taxon_id" not in df_eval: df_eval["taxon_id"]=df_eval.truth_label.astype(str)
strat_rows=[]
for artifact in artifacts:
    frame=artifact["predictions"].copy()
    if "taxon_id" not in frame: frame["taxon_id"]=frame.truth_label.astype(str)
    audited=stratify_long_errors(frame)
    mode=str(frame.handle_short.iloc[0])
    for dimension,key in [("length_bin","by_length"),("ambiguity_bin","by_ambiguity"),("taxon","by_taxon")]:
        for label,stats in audited[key].items(): strat_rows.append({"variant":artifact["variant"],"handle_short":mode,"dimension":dimension,"stratum":label,**stats})
strat_csv=output_dir/"stratified_long_errors.csv"; pd.DataFrame(strat_rows).to_csv(strat_csv,index=False)
composition=compare_query_and_taxon_accuracy(df_eval,is_smoke_test=ctx.is_smoke_test)
logit_findings=[]; calibration_findings=[]
for variant in C02_LONG_VARIANTS:
    vframe=next(a["predictions"] for a in artifacts if a["variant"]==variant and set(a["predictions"]["handle_short"].astype(str))=={"pad"})
    inputs=[variant]+manifest_02.get("diagnostic_logits",[])
    if "window_logits" not in vframe or not vframe.window_logits.notna().any():
        logit_findings.append(build_diagnostic_finding(f"window influence for {variant}","leave-one-window-out",{"reason":"raw logits unavailable"},0,inputs,["raw logits unavailable"],status="INCONCLUSIVE"))
        calibration_findings.append(build_diagnostic_finding(f"calibration for {variant}","15-bin ECE and NLL",{"reason":"raw logits unavailable"},0,inputs,["raw logits unavailable"],status="INCONCLUSIVE"))
    else:
        measured=diagnose_window_logits(vframe.window_logits.dropna().iloc[0],int(vframe.truth_label.iloc[0]),is_smoke_test=ctx.is_smoke_test)
        status="INCONCLUSIVE" if ctx.is_smoke_test else measured["status"]
        logit_findings.append(build_diagnostic_finding(f"observed window influence for {variant}","leave-one-window-out prediction changes",measured,measured["n_windows"],inputs,measured.get("limitations",[]),status=status,is_smoke_test=ctx.is_smoke_test))
        cal=calibrate_mean_logits(vframe.window_logits.tolist(),vframe.truth_label.tolist(),is_smoke_test=ctx.is_smoke_test)
        calibration_findings.append(build_diagnostic_finding(f"calibration for {variant}","15-bin ECE and NLL from softmax(mean logits)",cal,cal.get("n",0),inputs,cal.get("limitations",[]),status="INCONCLUSIVE" if ctx.is_smoke_test else cal["status"],is_smoke_test=ctx.is_smoke_test))
composition_finding=build_diagnostic_finding("query-weighted and taxon-macro accuracy comparison","paired weighting comparison",composition,int(composition["n"]),[str(manifest02_path)],[composition["genome_attribution"]],status="INCONCLUSIVE" if ctx.is_smoke_test else composition["status"],is_smoke_test=ctx.is_smoke_test)
diagnosis={"status":"INCONCLUSIVE","claim":"operational long-sequence observations only","table_12_n":int(len(truth)),"composition":composition_finding,"window_influence":logit_findings,"calibration":calibration_findings,"limitations":["diagnostics are observational and do not establish architecture/transfer causality","genome identity is unknown unless supplied by an accession mapping"]}
diag_path=output_dir/"long_sequence_diagnostics.json"; diag_path.write_text(json.dumps(diagnosis,indent=2,default=str),encoding="utf-8")


## 4. Diagnóstico Causal do Colapso de ConvNeXt

Síntese documentada dos três mecanismos responsáveis pela discrepância observada na Tabela 12.


In [ ]:
print(json.dumps(diagnosis,indent=2,default=str))


## 5. Emissão do Manifesto de Comparação e Diagnóstico


In [ ]:
manifest_03={"notebook":"03_cnn_long_voter_compare.ipynb","long_sequence_protocol":"c02_audit_v1","run_id":run_id,"is_smoke_test":ctx.is_smoke_test,"predecessors":[str(manifest01_path),str(manifest02_path)],"table_12":table_12_data,"diagnosis":diagnosis,"artifacts":{"table_12_csv":str(t12_csv),"stratified_errors_csv":str(strat_csv),"diagnosis_json":str(diag_path)}}
manifest_path=output_dir/"manifest_03_voter_compare.json"; manifest_path.write_text(json.dumps(manifest_03,indent=2,default=str),encoding="utf-8")
print(f"Manifest: {manifest_path}")

from viral_cnn.notebook_runtime import write_notebook_manifest
write_notebook_manifest("long/03",ctx.data_root,ctx.is_smoke_test,metrics_summary={"table_rows":len(df_t12),"diagnostic_status":diagnosis["status"]},input_files=[str(manifest01_path),str(manifest02_path)],run_id=ctx.run_id,configuration={"protocol":"c02_audit_v1","variants":list(C02_LONG_VARIANTS)},outputs=[t12_csv,strat_csv,diag_path,manifest_path],output_json_path=output_dir/"run_manifest.json")
